<div style="display:flex;align-items:center;gap:18px;padding:14px 0 18px 0;border-bottom:3px solid #0a4dcc;">
<img src="../assets/logo_senati.png" width="210">
<div>
<h1 style="margin:0;color:#0a3f99;">Trabajo Final · Fundamentos y Algoritmia para Inteligencia Artificial</h1>
<p style="margin:5px 0 0 0;font-size:16px;"><strong>Ing. Gunar Castro Iñigo</strong></p>
</div>
</div>

## 6. Entrenamos el modelo de Machine Learning

Llegamos a la parte que integra todo lo trabajado anteriormente.

En esta ocasión vamos a construir un modelo de **regresión Ridge** para estimar el `promedio_final`. Elegimos regresión porque nuestra variable objetivo es numérica.

El modelo estará dentro de un pipeline junto con el preprocesamiento. De esta manera, primero se limpian y transforman los datos y después se realiza el entrenamiento.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Leemos el dataset.
df = pd.read_csv("../dataset/rendimiento_estudiantil_senati.csv")

# Separamos variables de entrada y variable objetivo.
X = df.drop(columns=["promedio_final", "rendimiento_categoria"])
y = df["promedio_final"]

### Reconstruimos el preprocesamiento

Aunque ya lo desarrollamos en el cuaderno 2, lo repetimos aquí para que este archivo pueda ejecutarse de manera independiente.

In [2]:
columnas_numericas = [
    "edad", "horas_estudio_semana", "asistencia_pct",
    "tareas_completadas_pct", "participacion_pct",
    "uso_plataforma_horas", "tutorias_mes",
    "promedio_prev", "examen_parcial", "proyecto_final"
]

columnas_categoricas = ["genero", "internet_calidad"]

proceso_numerico = Pipeline(steps=[
    ("imputar", SimpleImputer(strategy="median")),
    ("normalizar", StandardScaler())
])

proceso_categorico = Pipeline(steps=[
    ("imputar", SimpleImputer(strategy="most_frequent")),
    ("codificar", OneHotEncoder(handle_unknown="ignore"))
])

preprocesador = ColumnTransformer(transformers=[
    ("num", proceso_numerico, columnas_numericas),
    ("cat", proceso_categorico, columnas_categoricas)
])

### Creamos el modelo

Ridge es una regresión lineal con regularización. Para este proyecto es apropiada porque es fácil de interpretar y funciona bien como modelo base.

In [3]:
modelo = Ridge(alpha=1.0)

pipeline = Pipeline(steps=[
    ("preprocesamiento", preprocesador),
    ("modelo", modelo)
])

print("Pipeline listo para entrenar.")

Pipeline listo para entrenar.


### Entrenamos y hacemos predicciones

Usaremos el 80 % de los registros para entrenar y el 20 % para evaluar.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

pipeline.fit(X_train, y_train)
predicciones = pipeline.predict(X_test)

print("Modelo entrenado correctamente.")

Modelo entrenado correctamente.


### Evaluamos el modelo

Utilizaremos tres métricas:

- **MAE:** error absoluto medio. Mientras más pequeño, mejor.
- **RMSE:** penaliza más los errores grandes. Mientras más pequeño, mejor.
- **R²:** indica qué proporción de la variación del promedio final es explicada por el modelo. Mientras más cercano a 1, mejor.

In [5]:
mae = mean_absolute_error(y_test, predicciones)
rmse = np.sqrt(mean_squared_error(y_test, predicciones))
r2 = r2_score(y_test, predicciones)

print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²  : {r2:.4f}")

MAE : 0.7020
RMSE: 0.8782
R²  : 0.6644


### Comparamos algunos valores reales con las predicciones

Esta tabla nos permite ver ejemplos concretos. No buscamos que cada predicción sea idéntica al valor real, sino que el modelo se acerque razonablemente.

In [6]:
comparacion = pd.DataFrame({
    "Promedio real": y_test.values,
    "Promedio estimado": np.round(predicciones, 2)
})

comparacion.head(15)

,Promedio real,Promedio estimado
0,11.1,10.62
1,9.8,11.31
2,12.2,13.61
3,13.6,15.50
4,12.9,13.06
5,12.5,14.23
6,12.1,13.94
7,14.7,15.68
8,12.9,12.54
9,15.5,16.53


In [7]:
# Guardamos las métricas y predicciones como evidencia del proyecto.
metricas = pd.DataFrame({
    "metrica": ["MAE", "RMSE", "R2"],
    "valor": [round(mae, 4), round(rmse, 4), round(r2, 4)]
})

metricas.to_csv("../resultados/metricas_modelo.csv", index=False, encoding="utf-8-sig")
comparacion.to_csv("../resultados/predicciones_prueba.csv", index=False, encoding="utf-8-sig")

print("Resultados guardados correctamente.")

Resultados guardados correctamente.


### Interpretación del resultado

El modelo no pretende reemplazar una evaluación académica real. Su función en este proyecto es demostrar cómo los datos previamente preparados pueden utilizarse para construir una predicción.

Con un R² alrededor de 0.66 en este dataset, el modelo logra explicar una parte importante de la variación del promedio final. El resultado es razonable para un ejercicio académico con datos sintéticos y múltiples factores que influyen en el rendimiento.

### Cierre de esta etapa

Ya completamos el flujo técnico del proyecto: datos, preprocesamiento, matemáticas, estadística, gráficos y Machine Learning.

**En el último cuaderno vamos a responder las preguntas guía y redactar las conclusiones de manera clara y natural.**

---
**Ing. Gunar Castro Iñigo · SENATI**  
Este cuaderno forma parte de una secuencia de trabajo. Continúa con el siguiente archivo para mantener el orden del proyecto.